# Daily Challenge – Week 6, Day 2: Classifying Handwritten Digits with CNNs
Fully connected network vs. CNN on MNIST. Run in **Google Colab** (a GPU makes the CNN much faster).

## 1. Load the MNIST dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical   # (keras.utils.np_utils was removed in newer Keras versions)

tf.random.set_seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("x_train:", x_train.shape, "| y_train:", y_train.shape)
print("x_test: ", x_test.shape, "| y_test: ", y_test.shape)

## 2. Preprocess the data for a Fully Connected Neural Network

In [ ]:
x_train_flat = x_train.reshape(-1, 28 * 28).astype("float32") / 255.0    # 28x28 → 784, scaled to 0–1
x_test_flat = x_test.reshape(-1, 28 * 28).astype("float32") / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)
print("Flattened:", x_train_flat.shape, "| one-hot labels:", y_train_cat.shape)

## 3. Build and train a Fully Connected Neural Network

In [ ]:
dense_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
dense_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
dense_model.summary()

dense_history = dense_model.fit(x_train_flat, y_train_cat, epochs=5, batch_size=128, validation_split=0.1)
dense_loss, dense_acc = dense_model.evaluate(x_test_flat, y_test_cat, verbose=0)
print(f"Fully connected – test accuracy: {dense_acc:.4f}")

## 4. Preprocess the data for a Convolutional Neural Network
Conv2D layers expect images with a **channel** dimension: (number of images, height, width, channels). MNIST is greyscale, so 1 channel.

In [ ]:
x_train_cnn = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test_cnn = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
print("CNN input shape:", x_train_cnn.shape)
# The labels are the same one-hot vectors as before (y_train_cat, y_test_cat)

## 5. Build and train a Convolutional Neural Network

In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPool2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPool2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
cnn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()

cnn_history = cnn_model.fit(x_train_cnn, y_train_cat, epochs=5, batch_size=128, validation_split=0.1)
cnn_loss, cnn_acc = cnn_model.evaluate(x_test_cnn, y_test_cat, verbose=0)
print(f"CNN – test accuracy: {cnn_acc:.4f}")

## 6. Compare the performance

In [ ]:
print(f"Fully connected: accuracy {dense_acc:.4f} | parameters {dense_model.count_params():,} | errors {round((1 - dense_acc) * 10000)} of 10,000")
print(f"CNN:             accuracy {cnn_acc:.4f} | parameters {cnn_model.count_params():,} | errors {round((1 - cnn_acc) * 10000)} of 10,000")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(dense_history.history["val_accuracy"], marker="o", label="Fully connected")
axes[0].plot(cnn_history.history["val_accuracy"], marker="o", label="CNN")
axes[0].set_title("Validation accuracy per epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].bar(["Fully connected", "CNN"], [dense_acc, cnn_acc], color=["grey", "steelblue"])
axes[1].set_ylim(0.95, 1.0)
axes[1].set_title("Test accuracy")
for i, v in enumerate([dense_acc, cnn_acc]):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha="center")
plt.show()

**Observations:**
- Both models do well on MNIST, but the **CNN is typically more accurate**: around **99%** vs. **~98%** for the fully connected network. That difference means the CNN makes roughly **half as many mistakes**.
- The CNN achieves this with a **similar or smaller number of parameters**, because a convolution filter re-uses the same small set of weights across the whole image.
- **Why the CNN wins:** the fully connected network flattens the image into 784 unrelated numbers and loses the 2D structure, so it has to learn separately that a stroke in the top-left is the same pattern as one in the bottom-right. The CNN keeps the image's shape: its filters detect local patterns (edges, curves, loops) wherever they appear, and pooling makes it tolerant to small shifts in position. That's exactly what handwritten digits need.
- The trade-off is speed: convolutions cost more computation, so the CNN trains slower on a CPU. With a GPU the difference is small.